# VWAP Statistical Model Research

## Phase 2 — Statistical Research Framework

### Research objectives

- Establish a reproducible historical research environment.
- Validate historical dataset quality and provenance.
- Reconstruct causal VWAP and probability-band observations.
- Reproduce the existing solid RMS and Gaussian ghost baselines.
- Evaluate statistical calibration and predictive performance.
- Establish chronological validation and out-of-sample protocols.

### Research principles

The existing engine is treated as a reference baseline rather
than an assumed optimal model.

Historical datasets must remain unchanged. Timestamp
conventions, overlapping observations, missing candles and
mixed-frequency data must be investigated before modelling.

Candidate distributions and trading rules must be evaluated
without look-ahead bias or retrospective parameter selection.

## 1. Research environment and dataset inventory

In [ ]:

from pathlib import Path
import sys

import pandas as pd
import numpy as np


# Locate the repository root whether the notebook starts
# from the project directory or the notebooks directory.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "research_config.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.research_config import RESEARCH_CONFIG


DATA_DIR = RESEARCH_CONFIG["data_dir"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Python interpreter: {sys.executable}")
print(f"Instrument: {RESEARCH_CONFIG['instrument']}")
print(f"Timeframe: {RESEARCH_CONFIG['timeframe']}")
print(f"Baseline: {RESEARCH_CONFIG['baseline_dataset']}")


In [ ]:

dataset_paths = sorted(DATA_DIR.glob("*.csv"))

assert dataset_paths, "No historical CSV datasets found."

dataset_inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "columns": tuple(pd.read_csv(path, nrows=0).columns),
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
        }
        for path in dataset_paths
    ]
)

assert dataset_inventory["is_baseline"].sum() == 1, (
    "Expected exactly one baseline dataset."
)

print(f"Historical datasets: {len(dataset_inventory)}")
print(f"Total size: {dataset_inventory['size_mb'].sum():.2f} MB")

display(dataset_inventory)


## 2. Historical Dataset Registry and Provenance

Historical observations are separated into three categories:

- Legacy historical datasets retained for baseline reproduction
  and comparison.
- Immutable raw MT5 acquisitions with recorded source metadata
  and content-derived dataset identities.
- Validated research datasets produced only after independent
  timestamp and data-quality assessment.

External storage allows datasets to be shared across research
branches without duplication or inclusion in Git history.

Successful file-integrity verification does not establish
historical timezone accuracy, trading-session completeness
or eligibility for statistical modelling.

In [ ]:
from src.research_data_diagnostics import sha256_file

legacy_dataset_registry = pd.DataFrame(
    [
        {
            "filename": path.name,
            "dataset_type": "legacy_historical",
            "dataset_id": f"sha256:{sha256_file(path)}",
            "file_size_bytes": path.stat().st_size,
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
            "provenance_status": "not_independently_verified",
            "research_eligibility": "pending_validation",
        }
        for path in dataset_paths
    ]
)

display(legacy_dataset_registry)

In [ ]:
from src.research_dataset_registry import (
    discover_raw_datasets,
)

RAW_DATA_DIR = RESEARCH_CONFIG["raw_data_dir"]
VALIDATED_DATA_DIR = RESEARCH_CONFIG["validated_data_dir"]

print(f"External raw data: {RAW_DATA_DIR}")
print(f"Validated data: {VALIDATED_DATA_DIR}")

raw_dataset_records = discover_raw_datasets(
    RAW_DATA_DIR
)

raw_dataset_registry = pd.DataFrame(
    raw_dataset_records
)

print(
    f"Registered raw acquisitions: "
    f"{len(raw_dataset_registry)}"
)

if not raw_dataset_registry.empty:
    display(
        raw_dataset_registry[
            [
                "dataset_label",
                "symbol",
                "timeframe",
                "rows",
                "broker_server",
                "integrity_verified",
                "timestamp_alignment",
                "research_eligibility",
            ]
        ]
    )
else:
    print("No raw acquisitions registered.")

In [ ]:
if not raw_dataset_registry.empty:
    display(
        raw_dataset_registry[
            [
                "dataset_label",
                "dataset_id",
                "requested_start",
                "requested_end_exclusive",
                "observed_first_timestamp",
                "observed_last_timestamp",
                "acquisition_started_utc",
                "code_version",
                "working_tree_dirty",
            ]
        ]
    )

## 3. Historical Timestamp and Data Quality Investigation

Raw MT5 timestamps must not be assumed to represent genuine
UTC solely because they have been decoded as Unix timestamps.

This investigation evaluates:

- Historical trading coverage and intraday observation gaps.
- Broker-server timestamp conventions.
- Historical daylight-saving transitions.
- Agreement between newly acquired and legacy observations.

Historical broker announcements provide independent evidence
against which timestamp interpretations can be assessed.

Raw observations remain unchanged throughout this investigation.

In [ ]:
annual_coverage_records = []
daily_coverage_records = []

for record in raw_dataset_records:

    frame = pd.read_csv(
        record["csv_path"],
        usecols=["mt5_time_raw"],
    )

    timestamps = pd.to_datetime(
        frame["mt5_time_raw"],
        unit="s",
        utc=True,
    )

    # These are provisional raw clock labels, not verified UTC.
    year = pd.Timestamp(record["requested_start"]).year

    daily = timestamps.groupby(
        timestamps.dt.normalize()
    ).agg(
        first="min",
        last="max",
        rows="size",
        unique_minutes="nunique",
    )

    daily["span_minutes"] = (
        (daily["last"] - daily["first"])
        .dt.total_seconds() / 60
    ).astype(int) + 1

    daily["internal_unobserved_minutes"] = (
        daily["span_minutes"] - daily["unique_minutes"]
    )

    daily["year"] = year
    daily["month"] = daily.index.month
    daily["raw_date"] = daily.index.strftime("%Y-%m-%d")

    daily_coverage_records.append(
        daily.reset_index(drop=True)
    )

    annual_coverage_records.append({
        "year": year,
        "rows": len(frame),
        "first_raw_timestamp": timestamps.min(),
        "last_raw_timestamp": timestamps.max(),
        "observed_days": len(daily),
        "typical_daily_rows": daily["rows"].mode().iloc[0],
        "days_with_internal_gaps": (
            daily["internal_unobserved_minutes"] > 0
        ).sum(),
        "internal_unobserved_minutes": (
            daily["internal_unobserved_minutes"].sum()
        ),
        "most_common_open": (
            daily["first"].dt.strftime("%H:%M").mode().iloc[0]
        ),
        "most_common_last_candle": (
            daily["last"].dt.strftime("%H:%M").mode().iloc[0]
        ),
    })

    del frame, timestamps

annual_coverage = pd.DataFrame(
    annual_coverage_records
).sort_values("year")

daily_coverage = pd.concat(
    daily_coverage_records,
    ignore_index=True,
)

monthly_coverage = daily_coverage.pivot_table(
    index="month",
    columns="year",
    values="rows",
    aggfunc="sum",
).reindex(range(1, 13))

print("ANNUAL HISTORICAL COVERAGE")
display(annual_coverage)

print("MONTHLY CANDLE COUNTS")
display(monthly_coverage)

In [ ]:
# Independent historical FTMO trading announcements.
# Format: date, announced server close, GMT offset, source.

timestamp_references = [
    (
        "2023-05-29", "20:00", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-25-may-2023/",
    ),
    (
        "2023-11-23", "20:00", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-23-nov-2023/",
    ),
    (
        "2024-09-02", "20:00", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-29-aug-2024/",
    ),
    (
        "2024-11-28", "20:00", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-21-nov-2024/",
    ),
    (
        "2025-01-09", "16:30", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-9-jan-2025/",
    ),
    (
        "2025-06-19", "20:00", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-5-jun-2025/",
    ),
    (
        "2025-07-04", "20:15", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-3-jul-2025/",
    ),
    (
        "2026-01-19", "20:00", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-15-jan-2026/",
    ),
]

timestamp_evidence_records = []

for date, close, offset, source in timestamp_references:

    observations = daily_coverage.loc[
        daily_coverage["raw_date"].eq(date)
    ]

    # A candle is labelled by its opening time.
    expected_server_last = (
        pd.Timestamp(f"{date} {close}", tz="UTC")
        - pd.Timedelta(minutes=1)
    )

    expected_utc_last = (
        expected_server_last
        - pd.Timedelta(hours=offset)
    )

    if len(observations) != 1:
        observed_last = pd.NaT
        server_error = np.nan
        utc_error = np.nan
        interpretation = "unavailable_or_ambiguous"

    else:
        observed_last = observations["last"].iloc[0]

        server_error = (
            observed_last - expected_server_last
        ).total_seconds() / 60

        utc_error = (
            observed_last - expected_utc_last
        ).total_seconds() / 60

        if server_error == 0:
            interpretation = "exact_server_clock_match"

        elif utc_error == 0:
            interpretation = "exact_utc_match"

        else:
            interpretation = "no_exact_match"

    timestamp_evidence_records.append({
        "date": date,
        "broker_offset_hours": offset,
        "announced_server_close": close,
        "expected_server_last": (
            expected_server_last.strftime("%H:%M")
        ),
        "expected_utc_last": (
            expected_utc_last.strftime("%H:%M")
        ),
        "observed_raw_last": (
            observed_last.strftime("%H:%M")
            if pd.notna(observed_last) else None
        ),
        "server_error_minutes": server_error,
        "utc_error_minutes": utc_error,
        "interpretation": interpretation,
        "source": source,
    })

timestamp_evidence = pd.DataFrame(
    timestamp_evidence_records
)

display(
    timestamp_evidence.drop(columns="source")
)

In [ ]:
# FTMO's documented US daylight-saving transition dates.
# Applicable to our US100.cash weekday observations.

from src.research_data_diagnostics import (
    FTMO_US100_DST_DATES,
    ftmo_us100_utc_from_raw,
)

ftmo_dst_dates = FTMO_US100_DST_DATES

ftmo_ohlc_columns = ["open", "high", "low", "close"]

comparison_columns = ftmo_ohlc_columns + [
    "tick_volume",
    "spread",
    "real_volume",
]

raw_frames = []
clock_records = []

for record in raw_dataset_records:

    year = pd.Timestamp(record["requested_start"]).year

    # Retain 2021 externally, but exclude it from primary research.
    if year not in ftmo_dst_dates:
        continue

    frame = pd.read_csv(
        record["csv_path"],
        usecols=["mt5_time_raw"] + comparison_columns,
    )

    raw_clock = pd.to_datetime(
        frame["mt5_time_raw"],
        unit="s",
        utc=True,
    )

    candidate_utc = ftmo_us100_utc_from_raw(
        frame["mt5_time_raw"]
    )

    offset_hours = (
        raw_clock - candidate_utc
    ).dt.total_seconds() / 3600

    summer = offset_hours.eq(3)

    candidate_ny = candidate_utc.dt.tz_convert(
        RESEARCH_CONFIG["session_timezone"]
    )

    frame["raw_clock"] = raw_clock
    frame["candidate_utc"] = candidate_utc

    clock_records.append({
        "year": year,
        "winter_observations": int((~summer).sum()),
        "summer_observations": int(summer.sum()),
        "first_candidate_utc": candidate_utc.min(),
        "last_candidate_utc": candidate_utc.max(),
        "first_candidate_ny": candidate_ny.min(),
        "last_candidate_ny": candidate_ny.max(),
        "candidate_duplicate_timestamps": int(
            candidate_utc.duplicated().sum()
        ),
        "candidate_order_preserved": (
            candidate_utc.is_monotonic_increasing
        ),
    })

    raw_frames.append(frame)

clock_diagnostics = pd.DataFrame(clock_records)

display(clock_diagnostics)

raw_comparison_data = pd.concat(
    raw_frames,
    ignore_index=True,
)

del raw_frames

assert raw_comparison_data["raw_clock"].is_unique
assert raw_comparison_data["candidate_utc"].is_unique

# Additional independently documented 2022 holiday check.
thanksgiving_2022 = daily_coverage.loc[
    daily_coverage["raw_date"].isin([
        "2022-11-24",
        "2022-11-25",
    ]),
    ["raw_date", "first", "last", "rows"],
]

print("2022 Thanksgiving — announced server close: 20:00")
display(thanksgiving_2022)

In [ ]:
comparison_records = []

for path in dataset_paths:

    if "2021" in path.name:
        continue

    legacy = pd.read_csv(
        path,
    )

    legacy["datetime"] = pd.to_datetime(
        legacy["datetime"],
        utc=True,
        errors="raise",
    )

    assert legacy["datetime"].is_unique, (
        f"Duplicate timestamps in {path.name}"
    )

    shared_columns = [
        column
        for column in comparison_columns
        if column in legacy.columns
    ]

    assert set(ftmo_ohlc_columns).issubset(shared_columns)

    for interpretation in ["raw_clock", "candidate_utc"]:

        reference = raw_comparison_data[
            [interpretation] + shared_columns
        ].rename(
            columns={interpretation: "datetime"}
        )

        matched = legacy[
            ["datetime"] + shared_columns
        ].merge(
            reference,
            on="datetime",
            how="inner",
            suffixes=("_legacy", "_new"),
            validate="one_to_one",
        )

        result = {
            "dataset": path.name,
            "clock_interpretation": interpretation,
            "legacy_rows": len(legacy),
            "matched_rows": len(matched),
            "timestamp_overlap_pct": (
                100 * len(matched) / len(legacy)
            ),
        }

        if matched.empty:
            comparison_records.append(result)
            continue

        legacy_ohlc = matched[
            [
                f"{column}_legacy"
                for column in ftmo_ohlc_columns
            ]
        ].to_numpy(dtype=float)

        new_ohlc = matched[
            [
                f"{column}_new"
                for column in ftmo_ohlc_columns
            ]
        ].to_numpy(dtype=float)

        # Tolerance only accommodates floating-point noise.
        ohlc_agreement = np.isclose(
            legacy_ohlc,
            new_ohlc,
            rtol=0,
            atol=1e-8,
        ).all(axis=1)

        close_error = (
            matched["close_legacy"]
            - matched["close_new"]
        ).abs()

        result.update({
            "ohlc_matching_rows": int(
                ohlc_agreement.sum()
            ),
            "ohlc_agreement_pct": (
                100 * ohlc_agreement.mean()
            ),
            "ohlc_conflicting_rows": int(
                (~ohlc_agreement).sum()
            ),
            "median_abs_close_difference": (
                close_error.median()
            ),
            "p95_abs_close_difference": (
                close_error.quantile(0.95)
            ),
            "max_abs_close_difference": (
                close_error.max()
            ),
        })

        for column in [
            "tick_volume",
            "spread",
            "real_volume",
        ]:

            if column in shared_columns:

                agreement = matched[
                    f"{column}_legacy"
                ].eq(
                    matched[f"{column}_new"]
                )

                result[f"{column}_agreement_pct"] = (
                    100 * agreement.mean()
                )

        comparison_records.append(result)

legacy_clock_comparison = pd.DataFrame(
    comparison_records
)

print("LEGACY TIMESTAMP AND PRICE COMPARISON")

display(
    legacy_clock_comparison[
        [
            "dataset",
            "clock_interpretation",
            "legacy_rows",
            "matched_rows",
            "timestamp_overlap_pct",
            "ohlc_agreement_pct",
            "ohlc_conflicting_rows",
            "median_abs_close_difference",
            "p95_abs_close_difference",
        ]
    ].round(4)
)

print("2025 BASELINE — DETAILED COMPARISON")

display(
    legacy_clock_comparison.loc[
        legacy_clock_comparison["dataset"].eq(
            RESEARCH_CONFIG["baseline_dataset"]
        )
    ].T
)

In [ ]:
import json
from tempfile import TemporaryDirectory

from src.research_data_diagnostics import (
    diagnose_raw_candles,
    sha256_file,
)

VALIDATED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

validation_records = []

for record in raw_dataset_records:

    year = pd.Timestamp(
        record["requested_start"]
    ).year

    if year not in FTMO_US100_DST_DATES:
        continue

    if (
        record["symbol"] != "US100.cash"
        or record["broker_server"] != "FTMO-Server"
    ):
        raise ValueError("Unsupported acquisition source.")

    # Confirm the original acquisition remains unchanged.
    if sha256_file(record["csv_path"]) != record["sha256"]:
        raise ValueError("Raw dataset checksum mismatch.")

    # Preserve the original candle values as CSV text.
    frame = pd.read_csv(
        record["csv_path"],
        dtype=str,
        keep_default_na=False,
    )

    quality = diagnose_raw_candles(frame)

    integrity_checks = [
        "invalid_timestamp_rows",
        "non_minute_aligned_rows",
        "timestamp_mismatch_rows",
        "backward_timestamp_moves",
        "exact_duplicate_rows",
        "conflicting_timestamps",
        "invalid_ohlc_rows",
        "invalid_volume_spread_rows",
        "zero_tick_volume_rows",
    ]

    failures = {
        key: quality[key]
        for key in integrity_checks
        if quality[key] != 0
    }

    if failures:
        raise ValueError(f"{year}: {failures}")

    corrected = ftmo_us100_utc_from_raw(
        frame["mt5_time_raw"].astype("int64")
    )

    # Confirm every corrected candle had closed
    # before its original acquisition began.
    acquisition_start = pd.Timestamp(
        record["acquisition_started_utc"]
    )

    if (
        corrected + pd.Timedelta(minutes=1)
        > acquisition_start
    ).any():
        raise ValueError(
            f"Incomplete candle detected in {year}"
        )

    # Retain the original clock labels while making
    # datetime the corrected UTC research timestamp.
    frame = frame.rename(
        columns={"datetime": "raw_datetime"}
    )

    frame.insert(0, "datetime", corrected)

    label = record["dataset_label"].replace(
        "_raw_",
        "_utc_",
        1,
    )

    output_folder = VALIDATED_DATA_DIR / label
    csv_path = output_folder / f"{label}.csv"
    metadata_path = output_folder / f"{label}.json"

    provenance = {
        "dataset_label": label,
        "source_dataset_id": record["dataset_id"],
        "source_sha256": record["sha256"],
        "source_metadata_sha256": sha256_file(
            record["metadata_path"]
        ),
        "correction_source_sha256": sha256_file(
            PROJECT_ROOT / "src"
            / "research_data_diagnostics.py"
        ),
        "symbol": record["symbol"],
        "timeframe": record["timeframe"],
        "broker_server": record["broker_server"],
        "rows": len(frame),
        "first_utc": corrected.min().isoformat(),
        "last_utc": corrected.max().isoformat(),
        "timestamp_policy": "FTMO US DST GMT+2/GMT+3",
        "dst_transition_dates": list(
            FTMO_US100_DST_DATES[year]
        ),
        "timestamp_correction_applied": True,
        "raw_timestamps_preserved": True,
        "observations_removed": 0,
        "structural_integrity_passed": True,
        "session_coverage_verified": False,
        "research_eligibility": "pending_session_validation",
    }

    # Allow the notebook to be rerun without overwriting
    # an existing corrected dataset.
    if output_folder.exists():

        existing = json.loads(
            metadata_path.read_text(encoding="utf-8")
        )

        if (
            any(
                existing.get(key) != value
                for key, value in provenance.items()
            )
            or sha256_file(csv_path) != existing["sha256"]
        ):
            raise ValueError(
                f"Existing dataset requires review: {label}"
            )

        status = "already_verified"

    else:

        # Stage both files before publishing the directory.
        with TemporaryDirectory(
            dir=VALIDATED_DATA_DIR
        ) as temporary:

            staging = Path(temporary) / label
            staging.mkdir()

            staged_csv = staging / csv_path.name

            frame.to_csv(
                staged_csv,
                index=False,
            )

            provenance["sha256"] = sha256_file(
                staged_csv
            )

            provenance["dataset_id"] = (
                f"sha256:{provenance['sha256']}"
            )

            (staging / metadata_path.name).write_text(
                json.dumps(provenance, indent=2),
                encoding="utf-8",
            )

            staging.rename(output_folder)

        status = "exported"

    validation_records.append({
        "year": year,
        "dataset": label,
        "rows": len(frame),
        "first_utc": corrected.min(),
        "last_utc": corrected.max(),
        "status": status,
        "research_eligibility": (
            "pending_session_validation"
        ),
    })

    del frame

validated_dataset_inventory = pd.DataFrame(
    validation_records
)

display(validated_dataset_inventory)

## 4. Model-Independent Historical Coverage

Historical coverage is assessed independently of model
parameters and trading hypotheses.

All corrected observations from 2022–2026 are retained.

Daily coverage describes available observations and internal
timestamp gaps without assuming that every unobserved minute
represents a data error.

Research-window completeness is evaluated dynamically using
the requirements of each candidate model.

The original VWAP anchor, warmup, volatility estimators and
Gaussian parameters are retained exclusively as a reference
benchmark.

Chronological research partitions are established before
candidate model development.

In [ ]:
from src.research_sessions import (
    validate_timestamp_index,
    summarise_daily_coverage,
    assess_windows,
)

timestamp_parts = []
source_dates = []

for record in sorted(
    raw_dataset_records,
    key=lambda r: r["requested_start"],
):

    year = pd.Timestamp(
        record["requested_start"]
    ).year

    if year not in FTMO_US100_DST_DATES:
        continue

    label = record["dataset_label"].replace(
        "_raw_", "_utc_", 1
    )

    folder = VALIDATED_DATA_DIR / label

    csv_path = folder / f"{label}.csv"
    metadata_path = folder / f"{label}.json"

    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    if metadata["source_dataset_id"] != record["dataset_id"]:
        raise ValueError(
            f"Incorrect source provenance: {label}"
        )

    if not metadata["structural_integrity_passed"]:
        raise ValueError(
            f"Structural validation failed: {label}"
        )

    if sha256_file(csv_path) != metadata["sha256"]:
        raise ValueError(
            f"Corrected dataset checksum mismatch: {label}"
        )

    timestamps = pd.read_csv(
        csv_path,
        usecols=["datetime"],
    )["datetime"]

    timestamps = validate_timestamp_index(
        pd.to_datetime(
            timestamps,
            utc=True,
            errors="raise",
        )
    )

    if len(timestamps) != metadata["rows"]:
        raise ValueError(
            f"Row count mismatch: {label}"
        )

    timestamp_parts.append(timestamps)

    dates = timestamps.tz_convert(
        RESEARCH_CONFIG["session_timezone"]
    ).strftime("%Y-%m-%d").unique()

    source_dates.append(
        pd.DataFrame({
            "date": dates,
            "dataset_id": metadata["dataset_id"],
        })
    )

all_research_timestamps = validate_timestamp_index(
    timestamp_parts[0].append(timestamp_parts[1:])
)

local_timestamps = all_research_timestamps.tz_convert(
    RESEARCH_CONFIG["session_timezone"]
)

coverage_manifest = summarise_daily_coverage(
    all_research_timestamps,
    start_date=local_timestamps.min().date(),
    end_date=local_timestamps.max().date(),
    timezone=RESEARCH_CONFIG["session_timezone"],
)

daily_sources = pd.concat(
    source_dates,
    ignore_index=True,
).groupby("date")["dataset_id"].agg(
    lambda values: ";".join(sorted(set(values)))
)

coverage_manifest = coverage_manifest.merge(
    daily_sources.rename("source_dataset_ids"),
    left_on="date",
    right_index=True,
    how="left",
    validate="one_to_one",
)

coverage_manifest["year"] = pd.to_datetime(
    coverage_manifest["date"]
).dt.year

assert (
    coverage_manifest["observed_minutes"].sum()
    == len(all_research_timestamps)
)

print("MODEL-INDEPENDENT HISTORICAL COVERAGE")

display(
    coverage_manifest.groupby("year").agg(
        calendar_days=("date", "size"),
        observed_days=("has_observations", "sum"),
        total_candles=("observed_minutes", "sum"),
        internal_unobserved_minutes=(
            "internal_unobserved_minutes", "sum"
        ),
    )
)

print("TOTAL OBSERVATIONS")
print(len(all_research_timestamps))

In [ ]:
partition_summary = (
    coverage_manifest.loc[
        coverage_manifest["has_observations"]
    ]
    .groupby("split", sort=False)
    .agg(
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
        observed_days=("date", "size"),
        observations=("observed_minutes", "sum"),
    )
)

print("CHRONOLOGICAL RESEARCH PARTITIONS")

display(partition_summary)

manifest_path = (
    RESEARCH_CONFIG["research_data_root"]
    / "session_manifest.csv"
)

manifest_content = coverage_manifest.to_csv(
    index=False,
    lineterminator="\n",
).encode("utf-8")

if manifest_path.exists():

    if manifest_path.read_bytes() != manifest_content:
        raise ValueError(
            "Existing coverage manifest differs. "
            "Review the changes before replacing it."
        )

else:

    with manifest_path.open("xb") as file:
        file.write(manifest_content)

print("Coverage manifest verified.")
print("All corrected observations retained.")
print("No model-specific session exclusions applied.")